# ⚡ Electricity Market Morning-Briefing Agent

**Lecture: Agentic AI in Energy**

In this notebook we build, step by step, an **AI agent** that writes a morning briefing for the German–Luxembourg (DE-LU) electricity market. You give it a date; it decides by itself which data to fetch, which analyses to run and which charts to draw. Then it explains **what drove the electricity prices** that day and cites the evidence behind every claim.

```
                ┌──────────────────────────── Agent loop ────────────────────────────┐
  "Brief me on  │   LLM  ──decides──▶  tool call  ──▶  Python function  ──▶  result  │
   2026-05-01"  │    ▲                                                          │    │
  ─────────────▶│    └───────────────── result added to the conversation ◀──────┘    │──▶ Briefing with
                │         (repeat until the LLM has enough evidence)                 │    evidence IDs
                └────────────────────────────────────────────────────────────────────┘
  Tools:  📡 ENTSO-E data  ·  📊 analysis  ·  📈 charts  ·  📚 knowledge base search
```

| Part | What you do | ⏱ |
|---|---|---|
| 0–2 | Setup, API keys, pick your LLM | 5 min |
| 3 | Fetch data from the ENTSO-E Transparency Platform | 5 min |
| 4–6 | Build the tools: analysis, charts, knowledge base | 10 min |
| 7–8 | Tool schemas and the agent loop, the core of "agentic AI" | 10 min |
| 9–10 | Generate the briefing and ask follow-up questions | 10 min |
| 11 | Exercises | open |

**How to use:** run the cells in order (`Shift + Enter`). Cells marked ✏️ invite you to change something.

## 0 · Setup

This installs two packages: [`entsoe-py`](https://github.com/EnergieID/entsoe-py), a client for the ENTSO-E API, and `openai`, a generic client that we use for **all** LLM providers. It also downloads the course material (knowledge documents and cached data).

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/mitre7/electricity-market-agent.git"  # instructor: set this
REPO_DIR = "electricity-market-agent"

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "entsoe-py", "openai"], check=True)
    if not Path("knowledge").exists():
        if not Path(REPO_DIR).exists():
            subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
        os.chdir(REPO_DIR)

assert Path("knowledge").exists(), "Run this notebook from the repository folder (knowledge/ not found)."
print("✅ Working directory:", Path.cwd())

In [ ]:
import json, re, math, time
from collections import Counter
from datetime import date as Date

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

pd.set_option("display.width", 160)
TZ, ZONE = "Europe/Berlin", "DE_LU"
MTU_START = "2025-10-01"   # first delivery day with 15-minute day-ahead prices; earlier days are not supported
STEP_H = 0.25              # one 15-minute market time unit (MTU) in hours: MWh = MW x 0.25
CACHE_DIR, FIG_DIR, KB_DIR = Path("data/cache"), Path("figures"), Path("knowledge")
FIG_DIR.mkdir(exist_ok=True)

## 1 · API keys

Your keys should **never** be typed into the notebook, because notebooks are easily shared. The cell below looks for each key in this order:

1. **Colab Secrets** (recommended): click the 🔑 icon in the left sidebar → *Add new secret* → enter the name and value → switch on **Notebook access**.
2. An environment variable or a local `.env` file (when running outside Colab).
3. Otherwise a hidden input box asks you for it. The key is kept only in memory until the runtime restarts.

| Secret name | Where to get it |
|---|---|
| `ENTSOE_API_KEY` | Register on [transparency.entsoe.eu](https://transparency.entsoe.eu), then email `transparency@entsoe.eu` with subject *"Restful API access"*. Your token then appears under *My Account Settings*. **No key? Leave it empty: the notebook switches to cached data.** |
| `OPENAI_API_KEY` / `ANTHROPIC_API_KEY` / `GEMINI_API_KEY` / `XAI_API_KEY` / `NVIDIA_API_KEY` | Only the one for the provider you pick in step 2. |

In [ ]:
def get_secret(name):
    # 1) Colab Secrets
    if IN_COLAB:
        try:
            from google.colab import userdata
            value = userdata.get(name)
            if value:
                return value.strip()
        except Exception:          # secret missing or notebook access not granted
            pass
    # 2) environment variable / .env file
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass
    if os.environ.get(name):
        return os.environ[name].strip()
    # 3) hidden prompt
    from getpass import getpass
    return getpass(f"{name} (hidden; press Enter to skip): ").strip() or None


ENTSOE_API_KEY = get_secret("ENTSOE_API_KEY")
print("ENTSO-E key:", "✅ found" if ENTSOE_API_KEY else "❌ not set → cached data only")

## 2 · Choose your LLM

All five providers below offer an **OpenAI-compatible** chat API, so a single client works for all of them: only the `base_url`, the API key and the model name change. Tool calling uses the same format everywhere.

✏️ Set `PROVIDER` to the one you have a key for. Model names change often. If one is rejected, run the *list models* cell and pick another.

In [ ]:
PROVIDERS = {
    "openai":    {"base_url": None,                                    "model": "gpt-6.1-sol",
                  "key": "OPENAI_API_KEY",    "params": {}},
    "anthropic": {"base_url": "https://api.anthropic.com/v1/",         "model": "claude-opus-5-5",
                  "key": "ANTHROPIC_API_KEY", "params": {"max_tokens": 16000}},
    "gemini":    {"base_url": "https://generativelanguage.googleapis.com/v1beta/openai/", "model": "gemini-3.8-flash",
                  "key": "GEMINI_API_KEY",    "params": {}},
    "xai":       {"base_url": "https://api.x.ai/v1",                   "model": "grok-4.7",
                  "key": "XAI_API_KEY",       "params": {}},
    "nvidia":    {"base_url": "https://inference-api.nvidia.com/v1",       "model": "nvidia/nvidia/nemotron-3-ultra",
                  "key": "NVIDIA_API_KEY",    "params": {"max_tokens": 16000}},
}

PROVIDER = "gemini"   # ✏️ "openai" | "anthropic" | "gemini" | "xai" | "nvidia"
MODEL = "gemini-3.1-flash-lite"           # ✏️ optional override, e.g. "claude-sonnet-5-5" or "grok-4.5"

from openai import OpenAI

cfg = PROVIDERS[PROVIDER]
MODEL = MODEL or cfg["model"]
client = OpenAI(api_key=get_secret(cfg["key"]), base_url=cfg["base_url"], max_retries=3, timeout=180)


# Gemini's free tier often answers 503 (overloaded) / 429 (rate limit): fall back to other models.
FALLBACKS = ["gemini-3.1-flash-lite", "gemini-3.5-flash-lite"] if PROVIDER == "gemini" else []


def chat(messages, tools=None):
    # One LLM call. Every provider receives exactly the same request shape.
    from openai import InternalServerError, RateLimitError
    models = [MODEL] + [m for m in FALLBACKS if m != MODEL]
    for attempt, model in enumerate(models):
        kwargs = dict(model=model, messages=messages, **cfg["params"])
        if tools:
            kwargs.update(tools=tools, tool_choice="auto")
        try:
            return client.chat.completions.create(**kwargs)
        except (InternalServerError, RateLimitError) as e:
            print(f"⚠️ {model}: {type(e).__name__}, trying next model...")
            time.sleep(2 * (attempt + 1))
    raise RuntimeError("All models failed (overloaded or rate-limited). Wait a minute and retry.")


def strip_thinking(text):
    # Some open reasoning models put their chain of thought in <think> tags; hide it.
    return re.sub(r"<think>.*?</think>", "", text or "", flags=re.S).strip()


print(f"🤖 Provider: {PROVIDER} · Model: {MODEL}")

In [ ]:
# Smoke test: one plain LLM call (no tools yet)
reply = chat([{"role": "user", "content": "In one sentence: what is the merit order in electricity markets?"}])
print(strip_thinking(reply.choices[0].message.content))

In [ ]:
# Optional: list the models your key can use
try:
    print(sorted(m.id for m in client.models.list())[:60])
except Exception as e:
    print("Listing models is not supported here:", e)

### 🧪 Why do we need an agent at all?

Ask the bare LLM about a specific day. It has no access to market data, so it can only refuse or, worse, make up something plausible.

In [ ]:
# Interesting delivery days, all cached offline at 15-minute resolution.
INTERESTING_DAYS = {   # name: (date, why it is interesting)
    "winter_scarcity": ("2025-11-25", "Winter scarcity: highest daily average of the winter, peak 400 €/MWh"),
    "dst_spring":      ("2026-03-29", "Spring clock change: only 92 quarter-hours (23-hour day)"),
    "easter_sunday":   ("2026-04-05", "Easter Sunday: negative for 69 quarter-hours (17 h), daily average below zero"),
    "labour_day":      ("2026-05-01", "Labour Day holiday: price at the -500 €/MWh floor"),
    "summer_spike":    ("2026-06-24", "Summer weekday with an evening spike to ~747 €/MWh"),
    "autumn_peak":     ("2026-09-14", "Monday with a peak of ~740 €/MWh and a daily average of ~257 €/MWh"),
    "recent_high":     ("2026-11-04", "Recent autumn weekday with a high price level"),
}

DAY = "labour_day"    # ✏️ pick a name from INTERESTING_DAYS
CUSTOM_DATE = None    # ✏️ or any other date from 2025-10-01 on, e.g. "2026-10-29" (needs your ENTSO-E key)

DATE = CUSTOM_DATE or INTERESTING_DAYS[DAY][0]   # the delivery day we analyse throughout the notebook

for name, (d, why) in INTERESTING_DAYS.items():
    print(("👉" if d == DATE else "  "), f"{name:16} {d}  {why}")

In [ ]:
reply = chat([{"role": "user", "content":
               f"What drove German day-ahead electricity prices on {DATE}? Give the min and max price."}])
print(strip_thinking(reply.choices[0].message.content))

## 3 · Data layer: the ENTSO-E Transparency Platform

[ENTSO-E](https://transparency.entsoe.eu) publishes European power market data. For one delivery day in the `DE_LU` zone we need:

| Query (`entsoe-py`) | What it gives us |
|---|---|
| `query_day_ahead_prices` | Day-ahead price per 15-minute interval |
| `query_load_and_forecast` | Actual load and day-ahead load forecast |
| `query_wind_and_solar_forecast` | Day-ahead wind (on/offshore) and solar forecasts |
| `query_generation` | Actual generation per technology |
| `query_net_position` | Day-ahead net position (exports − imports) |

Since 1 October 2025 the day-ahead market clears in **15-minute** intervals (market time units, MTU). This notebook therefore only uses delivery days from that date on and keeps the native resolution: **96 rows per day** (92 or 100 on the DST change days). Nothing is averaged. Power is in MW, and the energy in one interval is MW × 0.25 h.

First, a single live API call, so you can see what the raw data looks like:

In [ ]:
from entsoe import EntsoePandasClient

entsoe_client = EntsoePandasClient(api_key=ENTSOE_API_KEY) if ENTSOE_API_KEY else None

if entsoe_client:
    start = pd.Timestamp(DATE).tz_localize(TZ)
    t0 = time.time()
    raw_prices = entsoe_client.query_day_ahead_prices(ZONE, start=start, end=start + pd.Timedelta(hours=3))
    print(f"⏱ {time.time() - t0:.1f}s · {len(raw_prices)} price points")
    display(raw_prices.head(8))
else:
    print("No ENTSO-E key: skipping the live call. The cached data below has the same shape.")

Now the full data function. `load_day()` decides **where** the data comes from:

- `DATA_MODE = "auto"` (default): use the offline cache when the date is cached (fast), otherwise call the live API.
- `"live"`: always call the API, and fall back to the cache on errors.
- `"cache"`: never call the API.

A full live fetch takes about **1–2 minutes** because ENTSO-E is slow, so results are also kept in memory. Dates before 2025-10-01 are rejected (they are hourly).

In [ ]:
DATA_MODE = "auto"   # ✏️ "auto" | "live" | "cache"


def day_bounds(day):
    if pd.Timestamp(day) < pd.Timestamp(MTU_START):
        raise ValueError(f"{day}: this notebook only supports delivery days from {MTU_START} (15-minute MTU)")
    start = pd.Timestamp(day).tz_localize(TZ)
    end = (pd.Timestamp(day) + pd.Timedelta(days=1)).tz_localize(TZ)   # 23/25 h on DST days
    return start, end


def _snake(name):
    return name.lower().replace("/", "_").replace("-", "_").replace(" ", "_").replace("__", "_")


def fetch_day_live(day):
    # Query ENTSO-E for one delivery day -> (15-min frame, price history of the previous 30 days).
    start, end = day_bounds(day)
    clip = lambda x: x[(x.index >= start) & (x.index < end)]
    prices = clip(entsoe_client.query_day_ahead_prices(ZONE, start=start, end=end))
    load   = clip(entsoe_client.query_load_and_forecast(ZONE, start=start, end=end))
    ws_fc  = clip(entsoe_client.query_wind_and_solar_forecast(ZONE, start=start, end=end))
    gen    = clip(entsoe_client.query_generation(ZONE, start=start, end=end))
    netpos = clip(entsoe_client.query_net_position(ZONE, start=start, end=end, dayahead=True))
    # 30 calendar days back (not 720 absolute hours, which would add a partial day across DST), but never before the 15-min era
    hist_start = max(pd.Timestamp(day) - pd.Timedelta(days=30), pd.Timestamp(MTU_START)).tz_localize(TZ)
    hist = entsoe_client.query_day_ahead_prices(ZONE, start=hist_start, end=end)
    hist = hist[(hist.index >= hist_start) & (hist.index < end)]

    if isinstance(gen.columns, pd.MultiIndex):           # keep generation, drop pumping consumption
        gen = gen.xs("Actual Aggregated", axis=1, level=1)
    gen = gen.rename(columns=lambda c: "gen_" + _snake(c))

    # All series are natively 15-minute. This only puts them on one regular 15-min grid
    # (forward-fill, max 3 steps) in case a series is ever published at a coarser resolution.
    h = lambda x: x.resample("15min").ffill(limit=3)
    frame = pd.DataFrame({
        "price_eur_mwh":             h(prices),
        "load_actual_mw":            h(load["Actual Load"]),
        "load_forecast_mw":          h(load["Forecasted Load"]),
        "solar_forecast_mw":         h(ws_fc.get("Solar")),
        "wind_onshore_forecast_mw":  h(ws_fc.get("Wind Onshore")),
        "wind_offshore_forecast_mw": h(ws_fc.get("Wind Offshore")),
        "net_position_mw":           h(netpos),          # > 0 export, < 0 import
    }).join(h(gen))
    frame.index.name = "time"
    return frame, hist.to_frame("price_eur_mwh")


_MEMO, DATA_SOURCE = {}, {}


def load_day(day):
    day = str(pd.Timestamp(day).date())
    day_bounds(day)                       # rejects dates before the 15-minute era
    if day in _MEMO:
        return _MEMO[day]
    cache_file = CACHE_DIR / f"{ZONE}_{day}.parquet"
    hist_file = CACHE_DIR / f"{ZONE}_{day}_price_history.parquet"
    cached = cache_file.exists() and hist_file.exists()
    use_live = entsoe_client is not None and (DATA_MODE == "live" or (DATA_MODE == "auto" and not cached))
    if use_live:
        try:
            print(f"   📡 fetching {day} from ENTSO-E (≈1–2 min)…")
            _MEMO[day] = fetch_day_live(day)
            DATA_SOURCE[day] = "ENTSO-E live API"
            return _MEMO[day]
        except Exception as e:
            print(f"   ⚠️ live fetch failed ({type(e).__name__}: {e}); trying cache")
    if cached:
        _MEMO[day] = (pd.read_parquet(cache_file), pd.read_parquet(hist_file))
        DATA_SOURCE[day] = "offline cache (ENTSO-E snapshot)"
        return _MEMO[day]
    raise ValueError(f"No data for {day}: not in cache and live ENTSO-E access unavailable. "
                     f"Cached dates: {cached_dates()}")


def cached_dates():
    return sorted(p.stem.split("_")[-1] for p in CACHE_DIR.glob(f"{ZONE}_????-??-??.parquet"))


print("Cached dates:", cached_dates())

In [ ]:
df, hist = load_day(DATE)
print("Source:", DATA_SOURCE[str(pd.Timestamp(DATE).date())], "·", df.shape)
df.iloc[:, :8].round(0).head(16)   # the first 4 hours = 16 quarter-hours

## 4 · Tools, part 1: deterministic analysis

**Design principle:** the LLM never does arithmetic on raw data. It calls **deterministic Python functions** that return small JSON summaries. Each result gets an **evidence ID** (`E1`, `E2`, …, figures `FIG1`, …, documents `DOC…`). The briefing must cite these IDs, so every claim can be traced back to its source.

Key derived quantities:
- **Residual load** = load − wind − solar, the demand that coal, gas, biomass, hydro and imports must cover. We compute it from the *forecasts*, because the day-ahead price was set on forecasts, and from *actuals*.
- **Forecast error** = actual − forecast.

In [ ]:
EVIDENCE = {}
_COUNTERS = Counter()


def register_evidence(prefix, tool, args, summary, **extra):
    _COUNTERS[prefix] += 1
    eid = f"{prefix}{_COUNTERS[prefix]}"
    EVIDENCE[eid] = {"tool": tool, "args": args, "summary": summary, **extra}
    return eid


def reset_evidence():
    EVIDENCE.clear()
    _COUNTERS.clear()


HOLIDAYS = {  # nationwide German public holidays
    "2025-01-01": "New Year", "2025-04-18": "Good Friday", "2025-04-21": "Easter Monday",
    "2025-05-01": "Labour Day", "2025-05-29": "Ascension Day", "2025-06-09": "Whit Monday",
    "2025-10-03": "German Unity Day", "2025-12-25": "Christmas Day", "2025-12-26": "Boxing Day",
    "2026-01-01": "New Year", "2026-04-03": "Good Friday", "2026-04-06": "Easter Monday",
    "2026-05-01": "Labour Day", "2026-05-14": "Ascension Day", "2026-05-25": "Whit Monday",
    "2026-10-03": "German Unity Day", "2026-12-25": "Christmas Day", "2026-12-26": "Boxing Day",
}

GEN_GROUPS = {   # ENTSO-E technology -> group used in summaries and charts
    "Wind":            ["gen_wind_onshore", "gen_wind_offshore"],
    "Solar":           ["gen_solar"],
    "Biomass & hydro": ["gen_biomass", "gen_hydro_run_of_river_and_pondage", "gen_hydro_water_reservoir",
                        "gen_geothermal", "gen_other_renewable"],
    "Lignite":         ["gen_fossil_brown_coal_lignite"],
    "Hard coal":       ["gen_fossil_hard_coal"],
    "Gas":             ["gen_fossil_gas", "gen_fossil_coal_derived_gas"],
}


def day_type(day):
    d = str(pd.Timestamp(day).date())
    if d in HOLIDAYS:
        return f"public holiday ({HOLIDAYS[d]}, {pd.Timestamp(d).day_name()})"
    return pd.Timestamp(d).day_name() + (" (weekend)" if pd.Timestamp(d).dayofweek >= 5 else " (weekday)")


def gen_by_group(df):
    out = pd.DataFrame({g: df[[c for c in cols if c in df]].sum(axis=1) for g, cols in GEN_GROUPS.items()})
    used = [c for cols in GEN_GROUPS.values() for c in cols]
    out["Storage & other"] = df[[c for c in df if c.startswith("gen_") and c not in used]].sum(axis=1)
    return out


def enrich(df):
    d = df.copy()
    d["wind_forecast_mw"] = d["wind_onshore_forecast_mw"] + d["wind_offshore_forecast_mw"]
    d["wind_actual_mw"] = d["gen_wind_onshore"] + d["gen_wind_offshore"]
    d["solar_actual_mw"] = d["gen_solar"]
    d["residual_load_forecast_mw"] = d["load_forecast_mw"] - d["wind_forecast_mw"] - d["solar_forecast_mw"]
    d["residual_load_actual_mw"] = d["load_actual_mw"] - d["wind_actual_mw"] - d["solar_actual_mw"]
    return d


r1 = lambda x: None if pd.isna(x) else round(float(x), 1) + 0.0   # + 0.0 turns -0.0 into 0.0
gw = lambda x: r1(x / 1000)
hhmm = lambda ts: ts.strftime("%H:%M")   # start of the 15-min interval

In [ ]:
def get_market_overview(date):
    # Key price statistics, 30-day comparison, fundamentals and generation mix for one day (15-min data).
    df, hist = load_day(date)
    d, p = enrich(df), df["price_eur_mwh"]
    peak = p[(p.index.hour >= 8) & (p.index.hour < 20)]
    offpeak = p.drop(peak.index)
    n_neg = int((p < 0).sum())

    hourly = p.resample("1h").mean()                      # what an hourly market would have shown
    inside = p.resample("1h").agg(["min", "max"])
    swing = inside["max"] - inside["min"]                 # price range inside each clock hour

    daily = hist["price_eur_mwh"].resample("1D").mean().dropna()
    prev = daily.iloc[:-1]                 # the (up to) 30 days before the target date
    mix = gen_by_group(df).sum()
    load_total = d["load_actual_mw"].sum()

    result = {
        "date": str(pd.Timestamp(date).date()),
        "day_type": day_type(date),
        "data_source": DATA_SOURCE.get(str(pd.Timestamp(date).date())),
        "resolution": f"15-minute intervals; {len(p)} intervals on this day; 'time' = start of the interval",
        "prices_eur_mwh": {
            "baseload_avg": r1(p.mean()), "peak_avg_08_20": r1(peak.mean()), "offpeak_avg": r1(offpeak.mean()),
            "min": {"value": r1(p.min()), "time": hhmm(p.idxmin())},
            "max": {"value": r1(p.max()), "time": hhmm(p.idxmax())},
            "negative_quarter_hours": n_neg, "negative_hours_total": r1(n_neg * STEP_H),
            "max_minus_min_spread": r1(p.max() - p.min()),
        },
        "what_hourly_averages_would_hide": {
            "max_hourly_average": {"value": r1(hourly.max()), "hour": hhmm(hourly.idxmax())},
            "min_hourly_average": {"value": r1(hourly.min()), "hour": hhmm(hourly.idxmin())},
            "largest_price_range_inside_one_hour": {"range": r1(swing.max()), "hour": hhmm(swing.idxmax())},
        },
        "vs_previous_30_days": {
            "days_in_window": len(prev),
            "mean_of_daily_avgs": r1(prev.mean()), "std_of_daily_avgs": r1(prev.std()),
            "z_score_of_this_day": r1((p.mean() - prev.mean()) / prev.std()),
            "rank_of_this_day": f"{int((daily >= p.mean()).sum())} of {len(daily)} (1 = most expensive)",
            "highest_daily_avg": r1(prev.max()), "lowest_daily_avg": r1(prev.min()),
        },
        "daily_avg_fundamentals_gw": {
            "load_actual": gw(d["load_actual_mw"].mean()),
            "wind_actual": gw(d["wind_actual_mw"].mean()),
            "solar_actual": gw(d["solar_actual_mw"].mean()),
            "residual_load_actual": gw(d["residual_load_actual_mw"].mean()),
            "net_position_dayahead (+export/-import)": gw(d["net_position_mw"].mean()),
        },
        "wind_plus_solar_share_of_load_pct": r1(100 * (d["wind_actual_mw"] + d["solar_actual_mw"]).sum() / load_total),
        "generation_mix_share_pct": {k: r1(100 * v / mix.sum()) for k, v in mix.items()},
    }
    pr = result["prices_eur_mwh"]
    result["evidence_id"] = register_evidence(
        "E", "get_market_overview", {"date": date},
        f"{result['date']}: base {pr['baseload_avg']} €/MWh, min {pr['min']['value']} @ {pr['min']['time']}, "
        f"max {pr['max']['value']} @ {pr['max']['time']}, {pr['negative_quarter_hours']} negative quarter-hours "
        f"({pr['negative_hours_total']} h)",
        source=result["data_source"])
    return result


def analyze_price_drivers(date):
    # How well does forecast residual load explain the 15-min price? Includes the interval table.
    d = enrich(load_day(date)[0])
    p, rl = d["price_eur_mwh"], d["residual_load_forecast_mw"] / 1000
    slope, intercept = np.polyfit(rl, p, 1)
    corr = float(np.corrcoef(rl, p)[0, 1])
    mix = gen_by_group(d)

    def snapshot(ts):
        return {"time": hhmm(ts), "price": r1(p[ts]), "load_forecast_gw": gw(d.loc[ts, "load_forecast_mw"]),
                "wind_forecast_gw": gw(d.loc[ts, "wind_forecast_mw"]), "solar_forecast_gw": gw(d.loc[ts, "solar_forecast_mw"]),
                "residual_load_forecast_gw": r1(rl[ts]), "net_position_gw": gw(d.loc[ts, "net_position_mw"]),
                "gas_gw": gw(mix.loc[ts, "Gas"]), "coal_gw": gw(mix.loc[ts, "Lignite"] + mix.loc[ts, "Hard coal"])}

    table = pd.DataFrame({
        "price": p.round(1), "load_fc_gw": (d["load_forecast_mw"] / 1000).round(1),
        "wind_fc_gw": (d["wind_forecast_mw"] / 1000).round(1), "solar_fc_gw": (d["solar_forecast_mw"] / 1000).round(1),
        "resid_fc_gw": rl.round(1), "gas_gw": (mix["Gas"] / 1000).round(1),
        "netpos_gw": (d["net_position_mw"] / 1000).round(1)})
    table.index = [hhmm(t) for t in table.index]
    table.index.name = "time"

    result = {
        "date": str(pd.Timestamp(date).date()),
        "price_vs_residual_load_forecast": {
            "pearson_correlation": round(corr, 2), "r_squared": round(corr ** 2, 2),
            "slope_eur_per_mwh_per_gw": r1(slope),
            "interpretation_hint": "corr > 0.8: fundamentals explain the day well; low corr: look for other drivers",
        },
        "cheapest_interval": snapshot(p.idxmin()),
        "most_expensive_interval": snapshot(p.idxmax()),
        "interval_table_15min": table.to_csv(),
    }
    result["evidence_id"] = register_evidence(
        "E", "analyze_price_drivers", {"date": date},
        f"price~residual load: r={corr:.2f}, slope {slope:.1f} €/MWh per GW; 15-min table",
        source=DATA_SOURCE.get(result["date"]))
    return result


def analyze_forecast_errors(date):
    # Day-ahead forecast errors (actual - forecast) for load, wind, solar and residual load, in GW.
    d = enrich(load_day(date)[0])
    pairs = {"load": ("load_actual_mw", "load_forecast_mw"), "wind": ("wind_actual_mw", "wind_forecast_mw"),
             "solar": ("solar_actual_mw", "solar_forecast_mw"),
             "residual_load": ("residual_load_actual_mw", "residual_load_forecast_mw")}
    out = {}
    for name, (a, f) in pairs.items():
        err = (d[a] - d[f]) / 1000
        out[name] = {"mean_bias_gw": r1(err.mean()), "mean_abs_error_gw": r1(err.abs().mean()),
                     "largest_error": {"time": hhmm(err.abs().idxmax()), "error_gw": r1(err[err.abs().idxmax()])}}
    out["load"]["mape_pct"] = r1(100 * ((d["load_actual_mw"] - d["load_forecast_mw"]).abs() / d["load_actual_mw"]).mean())
    rl_err = (d["residual_load_actual_mw"] - d["residual_load_forecast_mw"]) / 1000
    worst = rl_err.abs().idxmax()
    # neighbouring quarter-hours have almost the same error, so the top list is built on hourly means
    hourly = pd.DataFrame({"err": rl_err, "price": d["price_eur_mwh"]}).resample("1h").mean()
    top = hourly["err"].abs().sort_values(ascending=False).index[:3]
    result = {
        "date": str(pd.Timestamp(date).date()),
        "sign_convention": "error = actual - forecast; positive residual-load error = system tighter than planned",
        "errors": out,
        "largest_residual_load_error_interval": {"time": hhmm(worst), "error_gw": r1(rl_err[worst]),
                                                  "price": r1(d.loc[worst, "price_eur_mwh"])},
        "top3_residual_load_error_hours": [{"hour": hhmm(t), "mean_error_gw": r1(hourly.loc[t, "err"]),
                                            "avg_price": r1(hourly.loc[t, "price"])} for t in top],
    }
    result["evidence_id"] = register_evidence(
        "E", "analyze_forecast_errors", {"date": date},
        f"MAE load {out['load']['mean_abs_error_gw']} GW, wind {out['wind']['mean_abs_error_gw']} GW, "
        f"solar {out['solar']['mean_abs_error_gw']} GW", source=DATA_SOURCE.get(result["date"]))
    return result

Let's call a tool **by hand**. This JSON is exactly what the LLM will see later:

In [ ]:
print(json.dumps(get_market_overview(DATE), indent=2, ensure_ascii=False))

## 5 · Tools, part 2: charts

The agent can request six chart types (price vs. residual load, generation mix, forecast errors, price history, price ramps, and an illustrative battery). Each chart is saved as a PNG and registered as evidence (`FIG…`), and the final briefing shows the charts the agent cited. The chart style follows a few data-viz rules: one y-axis per panel (no dual axes), a fixed colour order, and recessive grid lines.

In [ ]:
SURFACE, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
OTHER_GRAY, POS, NEG = "#a3a29c", "#2a78d6", "#e34948"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": INK2, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "axes.axisbelow": True,
    "grid.color": GRID, "grid.linewidth": 0.8, "font.size": 10, "axes.titlesize": 11,
    "axes.titleweight": "bold", "axes.titlelocation": "left", "lines.linewidth": 2, "legend.frameon": False,
})
CHART_TYPES = {
    "price_vs_residual_load": "Day-ahead price (top) and forecast load / wind+solar / residual load (bottom)",
    "generation_mix": "Stacked actual generation by technology group with actual load",
    "forecast_errors": "15-minute forecast errors (actual - forecast) for load, wind and solar",
    "price_history": "Daily average price over the previous 30 days, with the target day highlighted",
    "price_ramps": "Change in price versus the previous quarter-hour: shows the steepest up and down ramps",
    "battery_windows": "Illustrative 2 h battery: cheapest quarter-hours to charge, most expensive to discharge, and the resulting gross margin",
}


def _hour_axis(ax, idx, step=12):          # one tick every 12 quarter-hours = every 3 hours
    ax.set_xticks(range(0, len(idx), step))
    ax.set_xticklabels([hhmm(t) for t in idx][::step])
    ax.set_xlim(-0.5, len(idx) - 0.5)


def _label_end(ax, x, y, text, color=INK2):
    ax.annotate(text, (x, y), xytext=(6, 0), textcoords="offset points", va="center", fontsize=9, color=color)


def plot_price_vs_residual_load(d, day):
    x = np.arange(len(d))
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True, gridspec_kw={"hspace": 0.25})
    p = d["price_eur_mwh"]
    a1.step(x, p, where="post", color=SERIES[0])           # each price is valid for one 15-min block
    a1.axhline(0, color=INK2, lw=0.8)
    for ts, va in [(p.idxmin(), "top"), (p.idxmax(), "bottom")]:
        i = d.index.get_loc(ts)
        a1.annotate(f"{p[ts]:.0f} €/MWh @ {hhmm(ts)}", (i, p[ts]), xytext=(0, -10 if va == "top" else 8),
                    textcoords="offset points", ha="center", va=va, fontsize=9, color=INK)
    a1.margins(y=0.18)
    a1.set_title(f"DE-LU day-ahead price, 15-min · {day}")
    a1.set_ylabel("€/MWh")
    lines = [("Load forecast", d["load_forecast_mw"], OTHER_GRAY),
             ("Wind + solar forecast", d["wind_forecast_mw"] + d["solar_forecast_mw"], SERIES[2]),
             ("Residual load forecast", d["residual_load_forecast_mw"], SERIES[1])]
    for name, s, c in lines:
        a2.plot(x, s / 1000, color=c, label=name)
    a2.axhline(0, color=INK2, lw=0.8)
    a2.set_title("Day-ahead forecasts")
    a2.set_ylabel("GW")
    a2.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=9)
    _hour_axis(a2, d.index)
    return fig


def plot_generation_mix(d, day):
    x = np.arange(len(d))
    mix = gen_by_group(d) / 1000
    colors = SERIES[:len(mix.columns) - 1] + [OTHER_GRAY]
    fig, ax = plt.subplots(figsize=(9, 4.8))
    ax.stackplot(x, mix.T.values, labels=mix.columns, colors=colors, edgecolor=SURFACE, linewidth=1)
    ax.plot(x, d["load_actual_mw"] / 1000, color=INK, lw=2, label="Load (actual)")
    ax.set_title(f"Actual generation by technology · {day}")
    ax.set_ylabel("GW")
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=9)
    _hour_axis(ax, d.index)
    fig.tight_layout()
    return fig


def plot_forecast_errors(d, day):
    x = np.arange(len(d))
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.6), sharey=True)
    for ax, (name, a, f) in zip(axes, [("Load", "load_actual_mw", "load_forecast_mw"),
                                       ("Wind", "wind_actual_mw", "wind_forecast_mw"),
                                       ("Solar", "solar_actual_mw", "solar_forecast_mw")]):
        err = (d[a] - d[f]) / 1000
        ax.bar(x, err, color=np.where(err >= 0, POS, NEG), width=0.9)
        ax.axhline(0, color=INK2, lw=0.8)
        ax.set_title(f"{name}: actual − forecast")
        _hour_axis(ax, d.index, step=16)       # one tick every 4 hours
    axes[0].set_ylabel("GW")
    fig.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=POS), plt.Rectangle((0, 0), 1, 1, color=NEG)],
               labels=["actual above forecast", "actual below forecast"], loc="lower center", ncols=2, fontsize=9)
    fig.suptitle(f"Day-ahead forecast errors · {day}", x=0.01, ha="left", fontweight="bold")
    fig.tight_layout(rect=(0, 0.08, 1, 1))
    return fig


def plot_price_history(hist, day):
    daily = hist["price_eur_mwh"].resample("1D").agg(["mean", "min", "max"]).dropna()
    daily.index = daily.index.tz_localize(None)          # plot local calendar days
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.fill_between(daily.index, daily["min"], daily["max"], color="#cde2fb", lw=0, label="daily min–max")
    ax.plot(daily.index, daily["mean"], color=SERIES[0], marker="o", markersize=3, label="daily average")
    ax.axhline(daily["mean"].iloc[:-1].mean(), color=INK2, lw=1, ls="--", label="30-day mean")
    last = daily.iloc[-1]
    ax.plot(daily.index[-1], last["mean"], "o", ms=10, color=SERIES[1], mec=SURFACE, mew=2, label=str(day))
    ax.annotate(f"{last['mean']:.0f} €/MWh", (daily.index[-1], last["mean"]), xytext=(-10, -14),
                textcoords="offset points", ha="right", fontsize=9, color=INK)
    ax.axhline(0, color=INK2, lw=0.8)
    ax.set_title(f"Daily day-ahead prices, previous 30 days → {day}")
    ax.set_ylabel("€/MWh")
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=9)
    ax.xaxis.set_major_locator(matplotlib.dates.DayLocator(interval=5))
    ax.xaxis.set_major_formatter(matplotlib.dates.DateFormatter("%d %b"))
    ax.margins(y=0.12)
    return fig


def plot_price_ramps(d, day):
    x = np.arange(len(d))
    dp = d["price_eur_mwh"].diff().fillna(0)
    fig, ax = plt.subplots(figsize=(9, 3.6))
    ax.bar(x, dp, color=np.where(dp >= 0, POS, NEG), width=0.9)
    ax.axhline(0, color=INK2, lw=0.8)
    ax.margins(y=0.25)
    for ts in [dp.idxmax(), dp.idxmin()]:
        i = d.index.get_loc(ts)
        ax.annotate(f"{dp[ts]:+.0f} @ {hhmm(ts)}", (i, dp[ts]), xytext=(0, 6 if dp[ts] > 0 else -6),
                    textcoords="offset points", ha="center", va="bottom" if dp[ts] > 0 else "top", fontsize=9, color=INK)
    ax.set_title(f"Price change vs. previous quarter-hour · {day}")
    ax.set_ylabel("€/MWh per 15 min")
    _hour_axis(ax, d.index)
    return fig

def plot_battery_windows(d, day, hours=2, eff=0.9):
    # Illustrative only: perfect foresight, fixed daily cycle, no fees or degradation.
    p, x = d["price_eur_mwh"], np.arange(len(d))
    n = int(hours / STEP_H)                                      # quarter-hours needed to (dis)charge
    charge, discharge = p.nsmallest(n).index, p.nlargest(n).index
    cost = p[charge].sum() * STEP_H                              # EUR per MW of battery power
    revenue = eff * p[discharge].sum() * STEP_H
    pos = {t: i for i, t in enumerate(d.index)}
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.step(x, p, where="post", color=SERIES[0])
    for t in charge:
        ax.axvspan(pos[t], pos[t] + 1, color=SERIES[2], alpha=0.35, lw=0)
    for t in discharge:
        ax.axvspan(pos[t], pos[t] + 1, color=SERIES[1], alpha=0.35, lw=0)
    ax.axhline(0, color=INK2, lw=0.8)
    ax.set_title(f"Illustrative {hours:g} h battery: where to charge and discharge · {day}")
    ax.set_ylabel("€/MWh")
    ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=SERIES[2], alpha=0.5), plt.Rectangle((0, 0), 1, 1, color=SERIES[1], alpha=0.5)],
              labels=[f"charge ({n} cheapest quarter-hours)", f"discharge ({n} most expensive)"],
              loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=9)
    ax.text(0.01, 0.92, f"≈ {revenue - cost:,.0f} € per MW per day (perfect foresight, {eff:.0%} round-trip efficiency)",
            transform=ax.transAxes, fontsize=9, color=INK2)
    _hour_axis(ax, d.index)
    fig.facts = {"assumptions": f"1 MW / {hours:g} MWh battery, {eff:.0%} round-trip efficiency, perfect foresight, one cycle: "
                                f"charge in the {n} cheapest and discharge in the {n} most expensive quarter-hours; no fees or degradation",
                 "avg_charge_price_eur_mwh": r1(p[charge].mean()), "avg_discharge_price_eur_mwh": r1(p[discharge].mean()),
                 "gross_margin_eur_per_mw_day": r1(revenue - cost)}
    return fig


def create_chart(date, chart_type):
    # Create one chart, save it as PNG and return its evidence ID.
    if chart_type not in CHART_TYPES:
        raise ValueError(f"Unknown chart_type {chart_type!r}; choose from {list(CHART_TYPES)}")
    day = str(pd.Timestamp(date).date())
    df, hist = load_day(day)
    d = enrich(df)
    fig = {"price_vs_residual_load": lambda: plot_price_vs_residual_load(d, day),
           "generation_mix": lambda: plot_generation_mix(d, day),
           "forecast_errors": lambda: plot_forecast_errors(d, day),
           "price_history": lambda: plot_price_history(hist, day),
                         "price_ramps": lambda: plot_price_ramps(d, day),
           "battery_windows": lambda: plot_battery_windows(d, day)}[chart_type]()
    facts = getattr(fig, "facts", None)          # numbers drawn into the chart, so the LLM can cite them
    path = FIG_DIR / f"{day}_{chart_type}.png"
    fig.savefig(path, dpi=110, bbox_inches="tight")
    plt.close(fig)
    summary = CHART_TYPES[chart_type] + (f" | {facts['gross_margin_eur_per_mw_day']} EUR/MW/day" if facts else "")
    eid = register_evidence("FIG", "create_chart", {"date": day, "chart_type": chart_type}, summary, path=str(path))
    result = {"evidence_id": eid, "chart_type": chart_type, "description": CHART_TYPES[chart_type],
              "note": "Chart saved; it will be shown below the briefing. Cite it by its evidence_id."}
    if facts:
        result["chart_facts"] = facts
    return result

In [ ]:
res = create_chart(DATE, "price_vs_residual_load")
print(res)
display(Image(EVIDENCE[res["evidence_id"]]["path"]))

In [ ]:
# Show every chart type for the selected day (✏️ change DATE above, or set CHART_DAY here)
CHART_DAY = DATE

df_c, hist_c = load_day(CHART_DAY)
d_c = enrich(df_c)
ALL_PLOTS = {
    "price_vs_residual_load": lambda: plot_price_vs_residual_load(d_c, CHART_DAY),
    "generation_mix":         lambda: plot_generation_mix(d_c, CHART_DAY),
    "forecast_errors":        lambda: plot_forecast_errors(d_c, CHART_DAY),
    "price_history":          lambda: plot_price_history(hist_c, CHART_DAY),
    "price_ramps":            lambda: plot_price_ramps(d_c, CHART_DAY),
    "battery_windows":        lambda: plot_battery_windows(d_c, CHART_DAY),
}

for name, make in ALL_PLOTS.items():
    display(Markdown(f"#### {name}\n{CHART_TYPES[name]}"))
    fig = make()
    plt.show()
    plt.close(fig)

### 📊 Compare all cached days

Not an agent tool, just a presentation aid: the 15-minute price of every day in `INTERESTING_DAYS` on **one shared y-axis**, so the differences between day types are visible (negative prices in red), followed by a summary table.

In [ ]:
def plot_day_comparison(days=None):
    items = list((days or INTERESTING_DAYS).items())
    ncols = 4
    nrows = -(-len(items) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(13, 3 * nrows), sharey=True)
    for ax, (name, (day, why)) in zip(axes.flat, items):
        p = load_day(day)[0]["price_eur_mwh"]
        x = np.arange(len(p))
        ax.step(x, p, where="post", color=SERIES[0], lw=1.5)
        ax.fill_between(x, p.values, 0, where=p.values < 0, step="post", color=NEG, alpha=0.4, lw=0)
        ax.axhline(0, color=INK2, lw=0.8)
        ax.set_title(f"{name}\n{day}", fontsize=9)
        ax.set_xticks(range(0, len(p), 24))
        ax.set_xticklabels([hhmm(t) for t in p.index][::24], fontsize=8)
    for ax in axes.flat[len(items):]:
        ax.axis("off")
    for ax in axes[:, 0]:
        ax.set_ylabel("€/MWh")
    fig.suptitle("Day-ahead price per quarter-hour, same y-axis for all days", x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    return fig


def day_summary(days=None):
    rows = []
    for name, (day, why) in (days or INTERESTING_DAYS).items():
        d = enrich(load_day(day)[0])
        p, rl = d["price_eur_mwh"], d["residual_load_forecast_mw"] / 1000
        rows.append({"day": name, "date": day, "base €/MWh": r1(p.mean()), "min": r1(p.min()), "max": r1(p.max()),
                     "negative hours": r1((p < 0).sum() * STEP_H),
                     "wind+solar % of load": r1(100 * (d["wind_actual_mw"] + d["solar_actual_mw"]).sum() / d["load_actual_mw"].sum()),
                     "avg residual load GW": r1(rl.mean()), "corr(price, resid. load)": round(float(np.corrcoef(rl, p)[0, 1]), 2)})
    return pd.DataFrame(rows).set_index("day")


display(plot_day_comparison())
plt.close("all")
display(day_summary())

## 6 · Tools, part 3: knowledge base (retrieval)

The folder `knowledge/` holds short Markdown documents with domain knowledge: the merit order, negative prices, Dunkelflaute, forecast errors, market facts and a briefing playbook. We split them into sections and rank the sections with **BM25**, a classic keyword-search algorithm. It needs no vector database and no embedding model, and it works offline.

✏️ Open the files in the 📁 file browser. You can add your own `.md` file and re-run this cell.

In [ ]:
TOKEN = re.compile(r"[a-z0-9äöüß]+")
tokenize = lambda text: TOKEN.findall(text.lower())

KB = []
for path in sorted(KB_DIR.glob("*.md")):
    text = path.read_text(encoding="utf-8")
    title = text.strip().splitlines()[0].lstrip("# ").strip()
    for section in re.split(r"\n(?=## )", text):
        heading = section.strip().splitlines()[0].lstrip("# ").strip()
        KB.append({"doc": path.name, "title": title, "section": heading, "text": section.strip()})
# index the document title + section heading twice: a cheap "the heading matters more" boost
KB_TOKENS = [tokenize(f"{c['title']} {c['section']} " * 2 + c["text"]) for c in KB]
DF = Counter(t for toks in KB_TOKENS for t in set(toks))
AVG_LEN = sum(map(len, KB_TOKENS)) / len(KB_TOKENS)


def bm25(query, toks, k1=1.5, b=0.75):
    tf, score = Counter(toks), 0.0
    for q in set(tokenize(query)):
        if q in tf:
            idf = math.log(1 + (len(KB) - DF[q] + 0.5) / (DF[q] + 0.5))
            score += idf * tf[q] * (k1 + 1) / (tf[q] + k1 * (1 - b + b * len(toks) / AVG_LEN))
    return score


def search_knowledge_base(query, top_k=3):
    # Return the top-k knowledge-base sections for a query, each with its own evidence ID.
    ranked = sorted(range(len(KB)), key=lambda i: bm25(query, KB_TOKENS[i]), reverse=True)[:top_k]
    hits = []
    for i in ranked:
        c = KB[i]
        eid = register_evidence("DOC", "search_knowledge_base", {"query": query},
                                f"{c['doc']} › {c['section']}", source=c["doc"])
        hits.append({"evidence_id": eid, "document": c["doc"], "section": c["section"], "text": c["text"]})
    return {"query": query, "results": hits}


print(f"{len(KB)} sections from {len({c['doc'] for c in KB})} documents")
for h in search_knowledge_base("why do prices become negative on holidays")["results"]:
    print(f"  {h['evidence_id']:6} {h['document']} › {h['section']}")

## 7 · Describing the tools to the LLM

The LLM never runs Python itself. It only sees a **name, a description and a JSON schema** for each tool, and it answers with *"please call tool X with arguments Y"*. Our code runs the function and sends the result back.

The descriptions are effectively part of the prompt: clear descriptions lead to better tool choices.

### How one tool call works

Suppose we ask: *"What was the minimum price on 2026-05-01?"*

| Step | Who | What happens |
|---|---|---|
| 1 | our code | sends the question **plus the `TOOLS` list** to the LLM |
| 2 | LLM | does not answer yet. It replies with a *tool call request*: |
| 3 | our code | `call_tool` finds the Python function by name, runs it and gets a result |
| 4 | our code | sends the result back as a `"role": "tool"` message |
| 5 | LLM | now has real data and writes the final answer, citing the evidence ID |

The request in step 2 is plain JSON. The LLM only *names* the function and fills in the arguments:

```json
{"name": "get_market_overview", "arguments": "{\"date\": \"2026-05-01\"}"}
```

Note that `arguments` is a **string** containing JSON, which is why `call_tool` starts with `json.loads`. The LLM can request several calls in one step, and it can also decide to call no tool at all.

### What the cell below defines

There are three separate things, and it helps to keep them apart:

1. **`TOOLS`: the menu the LLM sees.** A list of descriptions in the JSON-schema format that the OpenAI-compatible API expects (Gemini, Claude, Grok and the others accept the same format). It contains **no code**. Each entry has:
   - `name`: what the LLM writes when it wants the tool,
   - `description`: when and why to use it (the LLM chooses tools from this text),
   - `parameters`: the arguments, each with a `type` and a `description`. `required` lists the ones the LLM must always provide. An `enum` limits a value to a fixed list (here the allowed `chart_type` values).
2. **`TOOL_FUNCTIONS`: the kitchen.** A dictionary that maps the same names to the real Python functions: `{"get_market_overview": get_market_overview, ...}`. The names must match `TOOLS` exactly.
3. **`call_tool`: the waiter.** It receives the name and the JSON arguments from the LLM, looks up the function, and calls it with `TOOL_FUNCTIONS[name](**args)`. The `**args` unpacks `{"date": "2026-05-01"}` into `get_market_overview(date="2026-05-01")`. If anything fails, the error text is returned to the LLM as the result, so it can correct itself and try again.

Two small helpers keep the list short: `_tool(...)` builds one entry in the schema format, and `DATE_ARG` is the shared definition of the `date` argument, because three tools take the same one. The `{**DATE_ARG, "chart_type": ...}` syntax copies the `date` argument and adds a second one.

✏️ **Try it:** the last line prints the full schema of the first tool, which is exactly what the LLM receives.

In [ ]:
def _tool(name, description, properties, required):
    # Build one tool description in the JSON-schema format the API expects (no code runs here).
    return {"type": "function", "function": {"name": name, "description": description,
            "parameters": {"type": "object", "properties": properties, "required": required}}}


# Shared argument definition: three tools take the same "date" string.
DATE_ARG = {"date": {"type": "string", "description": "Delivery day in ISO format YYYY-MM-DD"}}

# 1) The menu the LLM sees: name + description + arguments of every tool.
TOOLS = [
    _tool("get_market_overview",
          "Key day-ahead price statistics for DE-LU on one day from 15-minute data (base/peak/off-peak, min/max with time, "
          "negative quarter-hours, what hourly averages would hide), comparison with the previous 30 days, daily average "
          "load/wind/solar/residual load/net position and the generation mix. Call this first.", DATE_ARG, ["date"]),
    _tool("analyze_price_drivers",
          "Explains the 15-minute prices with fundamentals: correlation and slope of price vs. forecast residual load, "
          "snapshots of the cheapest and most expensive interval, and a 15-minute table (price, load, wind, solar, "
          "residual load, gas output, net position).", DATE_ARG, ["date"]),
    _tool("analyze_forecast_errors",
          "Day-ahead forecast errors (actual - forecast) for load, wind, solar and residual load: bias, mean absolute "
          "error, the largest 15-minute miss and the 3 hours with the biggest residual-load error.", DATE_ARG, ["date"]),
    _tool("create_chart",
          "Create a chart for the briefing. Returns a FIG evidence ID to cite.",
          {**DATE_ARG, "chart_type": {"type": "string", "enum": list(CHART_TYPES),
                                      "description": "; ".join(f"{k}: {v}" for k, v in CHART_TYPES.items())}},
          ["date", "chart_type"]),
    _tool("search_knowledge_base",
          "Search the curated knowledge base on electricity market fundamentals (merit order, negative prices, "
          "Dunkelflaute/scarcity, forecast errors, German market facts and holidays, briefing playbook). "
          "Use it to give context and to interpret patterns.",
          {"query": {"type": "string", "description": "Natural-language search query"},
           "top_k": {"type": "integer", "description": "Number of sections to return (default 3)"}}, ["query"]),
]

# 2) The matching Python functions, looked up by the name the LLM writes.
TOOL_FUNCTIONS = {f.__name__: f for f in [get_market_overview, analyze_price_drivers, analyze_forecast_errors,
                                          create_chart, search_knowledge_base]}


# 3) Run the tool the LLM asked for. `arguments` arrives as a JSON string, e.g. '{"date": "2026-05-01"}'.
def call_tool(name, arguments):
    # Run a tool requested by the LLM. Errors go back to the LLM as text so it can recover.
    try:
        args = json.loads(arguments or "{}")        # JSON string -> dict
        result = TOOL_FUNCTIONS[name](**args)       # {"date": "x"} -> get_market_overview(date="x")
    except Exception as e:
        result = {"error": f"{type(e).__name__}: {e}"}
    return json.dumps(result, ensure_ascii=False, default=str)


print(json.dumps(TOOLS[0], indent=2))

## 8 · The agent loop ⭐

This is the core of an agent, and it is surprisingly small:

1. Send the conversation and the tool list to the LLM.
2. If the LLM requests tool calls, run them, append the results to the conversation and **go back to 1**.
3. If it answers with text instead, that text is the final answer.

The **system prompt** sets the agent's role, how it should work and the output format. The guardrails are: answer only from evidence, cite IDs, and admit what it cannot check.

### The conversation is a list of messages

The LLM has **no memory between calls**. Every time we call it, we send the *whole* conversation so far as a list of messages, and each message has a `role`:

| Role | Written by | Purpose |
|---|---|---|
| `system` | our code | the instructions (`SYSTEM_PROMPT`) |
| `user` | our code | the question |
| `assistant` | the LLM | its reply: either text, or a **request to call tools** |
| `tool` | our code | the **result** of one tool call, sent back to the LLM |

A **tool message** carries two things: `tool_call_id`, which must match the `id` of the request it answers (so the LLM knows which result belongs to which call, even when it requests several tools at once), and `content`, the result as a string. If the function failed, `content` holds the error text, so the LLM can read it and try again.

### What `messages` looks like after one loop iteration

```python
[
  {"role": "system",    "content": "You are a senior electricity market analyst..."},
  {"role": "user",      "content": "Prepare the morning briefing for 2026-05-01"},

  # the LLM answers with a request, not with text
  {"role": "assistant", "tool_calls": [
      {"id": "call_1", "function": {"name": "get_market_overview",  "arguments": "{\"date\": \"2026-05-01\"}"}},
      {"id": "call_2", "function": {"name": "analyze_price_drivers", "arguments": "{\"date\": \"2026-05-01\"}"}}]},

  # our code ran both functions and answers each request, matched by id
  {"role": "tool", "tool_call_id": "call_1", "content": "{\"prices_eur_mwh\": {\"min\": {\"value\": -500.0, ...}}}"},
  {"role": "tool", "tool_call_id": "call_2", "content": "{\"correlation\": -0.97, ...}"},

  # next iteration: the LLM sees all of the above and writes text -> no tool calls -> the loop ends
  {"role": "assistant", "content": "### ⚡ DE-LU Morning Briefing - 2026-05-01 ..."},
]
```

In `run_agent` below, the line `messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})` creates the tool messages. Because the list is re-sent on every iteration, the data from earlier tool calls stays available to the LLM, and follow-up questions work by passing the same `conversation` back in.

✏️ After running the agent, try `conversation[2]`, `conversation[3]` and so on to see these messages for real.

In [ ]:
SYSTEM_PROMPT = '''You are a senior electricity market analyst. You write the morning briefing on the German-Luxembourg (DE-LU) day-ahead market for traders.

How to work:
- Gather evidence with the tools before writing. A good briefing usually needs get_market_overview, analyze_price_drivers, analyze_forecast_errors, at least one search_knowledge_base query to interpret the pattern you see, and 2-3 charts (create_chart). Request several tools in one step where possible.
- The data has 15-minute resolution (one price per quarter-hour, 96 per day; 92 or 100 on DST days). A time such as 13:15 is the START of that quarter-hour. Convert counts to hours with 4 quarter-hours = 1 hour, and say when a spike or negative price lasted only a few quarter-hours.
- Every number in the briefing must come from a tool result. Cite the evidence ID in square brackets right after the claim, e.g. "the minimum was -499.0 EUR/MWh at 13:15 [E1]". Cite knowledge with DOC ids and charts with FIG ids.
- Explain causes, not only numbers: link prices to residual load, wind and solar, demand (day type) and imports/exports.
- If something cannot be checked with the tools (gas or CO2 prices, plant outages, news), say so explicitly instead of guessing.

Format the briefing in Markdown, under about 350 words:
### ⚡ DE-LU Morning Briefing - <date>
**Headline:** one sentence
**Key numbers** - bullet points
**What drove prices** - 2 to 4 bullet points
**Forecast errors** - 1 to 3 bullet points
**Context** - one or two sentences from the knowledge base
**Watch list** - 2 or 3 bullet points
**Caveats** - one or two bullet points'''


def _preview(text, n=160):
    text = " ".join(str(text).split())
    return text if len(text) <= n else text[:n] + " …"


def run_agent(user_message, history=None, max_steps=12, verbose=True):
    # Run the tool-calling loop. Returns (final_answer, conversation) so you can ask follow-ups.
    if history is None:
        reset_evidence()
        history = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages = history + [{"role": "user", "content": user_message}]

    for step in range(1, max_steps + 1):
        response = chat(messages, tools=TOOLS)
        msg = response.choices[0].message

        assistant = {"role": "assistant"}
        if msg.content:
            assistant["content"] = msg.content
        if msg.tool_calls:  # model_dump keeps provider-specific fields (e.g. Gemini thought signatures)
            assistant["tool_calls"] = [tc.model_dump(exclude_none=True) for tc in msg.tool_calls]
        messages.append(assistant)

        if not msg.tool_calls:                                   # no more tools → final answer
            if verbose:
                print(f"✅ step {step}: final answer ({len(msg.content or '')} characters)")
            return strip_thinking(msg.content), messages

        if verbose:
            thought = strip_thinking(msg.content)
            print(f"🧠 step {step}" + (f": {_preview(thought)}" if thought else ""))
        for tc in msg.tool_calls:
            if verbose:
                print(f"   🔧 {tc.function.name}({tc.function.arguments})")
            result = call_tool(tc.function.name, tc.function.arguments)
            if verbose:
                print(f"      ↳ {_preview(result)}")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

    messages.append({"role": "user", "content": "Step limit reached. Write the best answer you can now, without tools."})
    final = chat(messages).choices[0].message.content
    messages.append({"role": "assistant", "content": final})
    return strip_thinking(final), messages


def show_briefing(text):
    # Render the briefing, the charts it cites, and an evidence appendix.
    display(Markdown(text))
    cited = list(dict.fromkeys(re.findall(r"\b((?:E|FIG|DOC)\d+)\b", text)))
    for eid in cited:
        if eid.startswith("FIG") and eid in EVIDENCE:
            display(Markdown(f"**[{eid}]** {EVIDENCE[eid]['summary']}"))
            display(Image(EVIDENCE[eid]["path"]))
    rows = [{"id": k, "cited": "✓" if k in cited else "", "tool": v["tool"], "arguments": json.dumps(v["args"]),
             "summary": v["summary"], "source": v.get("source", "")} for k, v in EVIDENCE.items()]
    display(Markdown("#### 🔎 Evidence log"))
    display(pd.DataFrame(rows).set_index("id"))
    uncited = [e for e in cited if e not in EVIDENCE]
    if uncited:
        print("⚠️ The briefing cites evidence IDs that do not exist:", uncited)

## 9 · Generate the morning briefing 🚀

Watch the trace: the LLM decides which tools to call, in which order and with which arguments. We did **not** hard-code this sequence.

In [ ]:
briefing, conversation = run_agent(f"Prepare the morning briefing for the DE-LU day-ahead market on {DATE}.")
show_briefing(briefing)

## 10 · Ask follow-up questions

The conversation (including all tool results) is kept, so you can drill down. The agent may call more tools if it needs them.

In [ ]:
answer, conversation = run_agent(
    "Which hours would have been most attractive for charging and discharging a battery, and why? "
    "Use the 15-minute data.",   # ✏️ ask your own question
    history=conversation)
show_briefing(answer)

## 11 · Your turn ✏️

**A. Another day.** Set `DAY` (or `CUSTOM_DATE`) in section 2 and re-run the notebook from there, or call the agent directly with any of the `INTERESTING_DAYS`. Cached days load instantly; any other date from 2025-10-01 onwards needs your ENTSO-E key (earlier days are hourly and not supported). Try comparing a winter scarcity day with a sunny holiday.

In [ ]:
day2 = INTERESTING_DAYS["winter_scarcity"][0]   # ✏️ try "easter_sunday", "dst_spring", ...
briefing2, _ = run_agent(f"Prepare the morning briefing for the DE-LU day-ahead market on {day2}.")
show_briefing(briefing2)

**B. Switch the model.** Change `PROVIDER` / `MODEL` in step 2, re-run that cell, then re-run step 9. Do different models call different tools? Which briefing is better grounded?

**C. Add a tool.** Write `compare_dates(date_a, date_b)`, which returns the key differences between two days. Add it to `TOOLS` and `TOOL_FUNCTIONS`, then ask *"Why was 2026-05-01 so much cheaper than 2026-09-14?"*. A scaffold is below.

**D. Grow the knowledge base.** Add `knowledge/07_my_notes.md` (e.g. notes on gas prices or a specific event), re-run section 6 and check whether the agent cites it.

**E. Add a critic.** Make a second LLM call that receives the briefing and the evidence JSON and lists every number that is **not** supported by the evidence. This "self-check" pattern is common in production agents.

In [ ]:
def compare_dates(date_a, date_b):
    # TODO: use get_market_overview() for both dates and return the differences that matter
    a, b = get_market_overview(date_a), get_market_overview(date_b)
    return {
        "date_a": a, "date_b": b,
        # e.g. "baseload_diff": ..., "wind_diff_gw": ..., "solar_diff_gw": ...
    }

# TOOLS.append(_tool("compare_dates", "Compare two delivery days ...",
#                    {"date_a": DATE_ARG["date"], "date_b": DATE_ARG["date"]}, ["date_a", "date_b"]))
# TOOL_FUNCTIONS["compare_dates"] = compare_dates

---
### 📝 Takeaways
- **Agent = LLM + tools + a loop.** The LLM plans and decides, and deterministic code does the data work.
- **Grounding:** tools return compact, structured evidence, and evidence IDs make every claim auditable.
- **Robustness:** fallbacks (cached data), error messages returned to the LLM, and a step limit.
- **Provider-agnostic:** the same loop runs on OpenAI, Anthropic, Gemini, xAI and NVIDIA models.

*Data: ENTSO-E Transparency Platform. Prices in EUR/MWh, power in MW/GW, all times in Europe/Berlin. Resolution: 15-minute intervals (MTU) since 1 Oct 2025.*